In [1]:
import sys
print(sys.version)
print(sys.executable)

3.14.3 | packaged by conda-forge | (main, Feb  5 2026, 05:49:06) [GCC 14.3.0]
/home/is/frederikus-h/miniconda3/envs/lt-eval/bin/python


In [2]:
# import os
import re
import json
import Levenshtein
import edlib
from pathlib import Path

In [3]:
_HOME_PATH_ = Path.home()
# _HOME_PATH_ = Path("/Volumes/nlp-work9")

In [4]:
_fp_transcript_asr_output_all_text = _HOME_PATH_ / "_self_research/24.10.lecture_translate/lectrans-testset/25.11.transcript_translate/transcript_asr_output_by_sbj_all_text.json"
_fp_transcript_anntd_output_all_text = _HOME_PATH_ / "_self_research/24.10.lecture_translate/lectrans-testset/25.11.transcript_translate/transcript_translate_annotated_by_sbj_all_text.json"
assert _fp_transcript_asr_output_all_text.exists() and _fp_transcript_anntd_output_all_text.exists()

In [5]:
with open(_fp_transcript_asr_output_all_text, "r") as f:
    transcripts_asr = json.load(f)
with open(_fp_transcript_anntd_output_all_text, "r") as f:
    transcripts_anntd = json.load(f)
assert transcripts_asr.keys() == transcripts_anntd.keys()
subjects = list(transcripts_asr.keys())

In [109]:
# sbj = subjects[0]
# t_asr = {k: v["asr"] for k, v in transcripts_asr[sbj].items()}
# t_anntd = {k: v["en"] for k, v in transcripts_anntd[sbj].items()}
# tid = list(t_asr.keys())[0]
#
# sbj, tid, t_asr[tid], t_anntd[tid]

('anat-1/LQ7IXyQSXBk',
 '0:34:15.868',
 'is an osteon. So, this is an osteon, is a structural unit of the compact bone.',
 'So, this is an osteon, is a structural unit of the compact bone.')

In [6]:
for sbj in subjects:
    t_asr = {k: v["asr"] for k, v in transcripts_asr[sbj].items()}
    t_anntd = {k: v["en"] for k, v in transcripts_anntd[sbj].items()}
    assert t_asr.keys() == t_anntd.keys(), f"Discrepancies of Time ID in subject: '{sbj}'"

---
##### Execute Alignment with Levenhstein Distance

In [ ]:
def get_aligned_asr(t_asr, t_anntd):
    combined_asr, combined_anntd = ' '.join(t_asr.values()), ' '.join(t_anntd.values())
    ops = iter(Levenshtein.opcodes(combined_asr, combined_anntd))

    aligned_asr = []    # aligned ASR text with the same time IDs as the original transcripts
    cur_asr_txt = ""    # placeholder for the currently alligned ASR text
    cum_idx_ann = -1    # cumulative index of the annotation
    op, asr_st, asr_ed, ann_st, ann_ed = next(ops)

    for tid, cur_ann in t_anntd.items():
        # cum_idx_asr += len(cur_asr) + 1
        cum_idx_ann += len(cur_ann) + 1

        try:
            while ann_st < cum_idx_ann:

                if op == "delete":
                    cur_asr_txt += combined_asr[asr_st:asr_ed]
                    op, asr_st, asr_ed, ann_st, ann_ed = next(ops)

                elif op in {"equal", "replace"}:
                    if ann_ed > cum_idx_ann:
                        cur_len = cum_idx_ann - ann_st
                        cur_asr_txt += combined_asr[asr_st : (asr_st + cur_len)]
                        asr_st += cur_len
                        ann_st += cur_len
                    else:
                        cur_asr_txt += combined_asr[asr_st:asr_ed]
                        op, asr_st, asr_ed, ann_st, ann_ed = next(ops)

                elif op == "insert":
                    if ann_ed > cum_idx_ann:
                        ann_st = cum_idx_ann
                    else:
                        op, asr_st, asr_ed, ann_st, ann_ed = next(ops)
        except StopIteration:
            op, asr_st, asr_ed, ann_st, ann_ed = "", asr_ed, asr_ed, ann_ed, ann_ed

        aligned_asr.append((tid, cur_asr_txt.strip()))
        cur_asr_txt = ""

    return dict(aligned_asr)

In [10]:
transcripts_aligned_asr = {}
for sbj in subjects:
    t_asr = {k: v["asr"] for k, v in transcripts_asr[sbj].items()}
    t_anntd = {k: v["en"] for k, v in transcripts_anntd[sbj].items()}
    aligned_asr = get_aligned_asr(t_asr, t_anntd)
    transcripts_aligned_asr[sbj] = {
        tid: {"aligned_asr": txt} for tid, txt in aligned_asr.items()
    }


In [11]:
_fp_transcript_aligned_asr_output_all_text = _fp_transcript_asr_output_all_text.with_suffix(".aligned.json")
with open(_fp_transcript_aligned_asr_output_all_text, "w", encoding="utf8") as f:
    json.dump(transcripts_aligned_asr, f, indent=2, ensure_ascii=False)

---
##### Execute Alignment with edlib Library

In [72]:
def get_aligned_asr_edlib(t_asr, t_anntd):
    combined_asr, combined_anntd = ' '.join(t_asr.values()), ' '.join(t_anntd.values())
    alignResult = edlib.align(combined_anntd, combined_asr, task='path', mode='NW')
    _ = list(filter(None, re.split(r"([IDX=])", alignResult['cigar'])))
    ops = iter(zip(map(int, _[::2]), _[1::2]))

    aligned_asr = []    # aligned ASR text with the same time IDs as the original transcripts
    cur_asr_txt = ""    # placeholder for the currently alligned ASR text
    cum_idx_ann = -1    # cumulative index of the annotation
    asr_st, ann_st = 0, 0    # current index respective to `combined_asr`, and `combined_anntd`
    cnt, op = next(ops)

    for tid, cur_ann in t_anntd.items():
        cum_idx_ann += len(cur_ann) + 1

        try:
            while ann_st < cum_idx_ann:

                if op == 'D':
                    cur_asr_txt += combined_asr[asr_st : (asr_st + cnt)]
                    # print(cnt, op, cum_idx_ann, '\n ', ann_st, cur_ann, '\n ', asr_st, f"({cnt})", combined_asr[asr_st : (asr_st + cnt)])
                    asr_st += cnt
                    cnt, op = next(ops)

                elif op in {'=', 'X'}:
                    if (ann_st + cnt) > cum_idx_ann:
                        cur_len = cum_idx_ann - ann_st
                        cur_asr_txt += combined_asr[asr_st : (asr_st + cur_len)]
                        # print(cnt, op, cum_idx_ann, '\n ', ann_st, cur_ann, '\n ', asr_st, f"({cur_len})", combined_asr[asr_st : (asr_st + cur_len)])
                        asr_st += cur_len
                        ann_st += cur_len
                        cnt -= cur_len
                    else:
                        cur_asr_txt += combined_asr[asr_st : (asr_st + cnt)]
                        # print(cnt, op, cum_idx_ann, '\n ', ann_st, cur_ann, '\n ', asr_st, f"({cnt})", combined_asr[asr_st : (asr_st + cnt)])
                        asr_st += cnt
                        ann_st += cnt
                        cnt, op = next(ops)

                elif op == 'I':
                    # print(cnt, op, cum_idx_ann, '\n   >>', ann_st, cur_ann)
                    if (ann_st + cnt) > cum_idx_ann:
                        # print("\t\t", ann_st, cnt, cum_idx_ann)
                        cur_len = cum_idx_ann - ann_st
                        ann_st += cur_len
                        cnt -= cur_len
                    else:
                        ann_st += cnt
                        cnt, op = next(ops)

        except StopIteration:
            cnt, op = 0, ''

        aligned_asr.append((tid, cur_asr_txt.strip()))
        cur_asr_txt = ""

    return dict(aligned_asr)

In [73]:
transcripts_aligned_asr = {}
for sbj in subjects:
    t_asr = {k: v["asr"] for k, v in transcripts_asr[sbj].items()}
    t_anntd = {k: v["en"] for k, v in transcripts_anntd[sbj].items()}
    aligned_asr = get_aligned_asr_edlib(t_asr, t_anntd)
    transcripts_aligned_asr[sbj] = {
        tid: {"aligned_asr": txt} for tid, txt in aligned_asr.items()
    }

In [75]:
_fp_transcript_aligned_asr_output_all_text = _fp_transcript_asr_output_all_text.with_suffix(".aligned_edlib.json")
with open(_fp_transcript_aligned_asr_output_all_text, "w", encoding="utf8") as f:
    json.dump(transcripts_aligned_asr, f, indent=2, ensure_ascii=False)

---
##### Compare the result from the two libraries

In [76]:
with open(_fp_transcript_asr_output_all_text.with_suffix(".aligned.json"), "r") as f:
    transcripts_asr_aligned_1 = json.load(f)

In [77]:
with open(_fp_transcript_asr_output_all_text.with_suffix(".aligned_edlib.json"), "r") as f:
    transcripts_asr_aligned_2 = json.load(f)

In [81]:
for sbj in transcripts_asr_aligned_1.keys():
    for tid in transcripts_asr_aligned_1[sbj].keys():
        a = transcripts_asr_aligned_1[sbj][tid]["aligned_asr"]
        b = transcripts_asr_aligned_2[sbj][tid]["aligned_asr"]
        if a != b:
            print(f"{sbj}  |  {tid}\n  >> \"{a}\"\n  >> \"{b}\"")

anat-2/IvK-UGOI5ZQ  |  0:08:15.548
  >> "And muscle tissue is in a continuous state of activity, even when you're not doing anything, when you're just sitting there, the muscles."
  >> "And muscle tissue is in a continuous state of activity, even when you're not doing anything, when you're just sitting there, the muscles. j"
anat-2/IvK-UGOI5ZQ  |  0:08:36.188
  >> "just for a muscle to have a baseline activity, it is creating the more you exercise, heat production is And you've all noticed that if you're, if it's cold and you're out running, once you start running, muscles when they're using cellular metabolism, that ATP, remember ATP, when ATP is released, you are released, you are generating heat with that too."
  >> "ust for a muscle to have a baseline activity, it is creating the more you exercise, heat production is And you've all noticed that if you're, if it's cold and you're out running, once you start running, muscles when they're using cellular metabolism, that ATP, remember 

---
##### Calculating Total Number of Edit Path

In [7]:
def calc_dp_val(a, b, f_a, f_b):
    if a > b:
        return b, f_b
    if a == b:
        return b, (f_a | f_b)
    return a, f_a


class Solution:
    def __init__(self):
        self._word1 = None
        self._word2 = None
        self.M = None
        self.N = None
        self.dp = None
        self.track = None

    def min_distance(self, word1: str, word2: str) -> int:
        M, N = len(word1), len(word2)
        c = 2 * (M + N)    # inf

        dp = [[c for _ in range(N+1)] for __ in range(M+1)]
        track = [[0 for _ in range(N+1)] for __ in range(M+1)]

        dp[0][0] = 0
        for i in range(M+1):
            for j in range(N+1):
                if (i > 0):    # delete
                    dp[i][j], track[i][j] = calc_dp_val(
                        dp[i][j], dp[i-1][j] + 1,
                        track[i][j], 1,
                    )

                if (j > 0):    # insert
                    dp[i][j], track[i][j] = calc_dp_val(
                        dp[i][j], dp[i][j-1] + 1,
                        track[i][j], 2,
                    )

                if (i > 0) and (j > 0):    # match / replace
                    dp[i][j], track[i][j] = calc_dp_val(
                        dp[i][j], dp[i-1][j-1] + (0 if word1[i-1] == word2[j-1] else 1),
                        track[i][j], 4,
                    )

        self._word1, self._word2 = word1, word2
        self.M, self.N = M, N
        self.dp = dp
        self.track = track
        return dp[M][N]

    def count_paths(self) -> int:
        M, N, track = self.M, self.N, self.track
        prev = [0] * (N+1)
        cur = [0] * (N+1)
        cur[0] = 1
        for i in range(M+1):
            for j in range(N+1):
                if track[i][j] & 1:
                    cur[j] += prev[j]
                if track[i][j] & 2:
                    cur[j] += cur[j-1]    # error means self.track is not computed properly
                if track[i][j] & 4:
                    cur[j] += prev[j-1]    # error means self.track is not computed properly

            cur, prev = [0] * (N+1), cur
        return prev[-1]


In [14]:
import time
from decimal import Decimal

In [16]:
# sols = {}
_t0 = time.perf_counter()
for sbj in transcripts_asr.keys():
    t_asr = {k: v["asr"] for k, v in transcripts_asr[sbj].items()}
    t_anntd = {k: v["en"] for k, v in transcripts_anntd[sbj].items()}
    cur_sol = Solution()
    _t1 = time.perf_counter()
    min_dist = cur_sol.min_distance(' '.join(t_asr.values()), ' '.join(t_anntd.values()))
    _t2 = time.perf_counter()
    # print(f"{_t2-_t1:.2f}s  (edit calculation)")
    path_cnt = cur_sol.count_paths()
    _t3 = time.perf_counter()
    # print(f"{_t3-_t2:.2f}s  (total: {_t3-_t1:.2f}s)")
    # sols[sbj] = cur_sol
    print(sbj, f"{Decimal(path_cnt):.3e}", min_dist, cur_sol.M, cur_sol.N, f"({_t3-_t1:.2f}s)", sep='\t')

print(f"Finished in {time.perf_counter()-_t0:.2f}s")

anat-1/LQ7IXyQSXBk	7.958e+58	1125	8401	9013	(27.50s)
anat-2/IvK-UGOI5ZQ	1.848e+55	1258	4670	5869	(9.78s)
bio-1/pUH2_SGnm9g	1.235e+17	359	9702	9961	(33.47s)
bio-2/dEh_Zk9d0sE	4.151e+21	346	8989	9165	(28.36s)
bio-3/u7AnKjGOEgI	1.656e+66	950	6281	6622	(14.55s)
bio-4/gb0arNyiLCI	3.336e+32	708	6061	6368	(13.43s)
dental/uLz-Ur_Drk4	6.548e+38	875	7509	8332	(21.95s)
intmd_microeconomics/rhOpABGewYA	1.755e+103	2085	5681	7616	(15.96s)
intro_to_cs_programming_python/tsMrUdynbQQ	3.017e+20	441	7655	8072	(25.82s)
intro_to_neural_computation/K1pxJVdqlxw	8.122e+44	873	7290	7914	(23.29s)
intro_to_physics_20century/Fce9xf_kjyc	1.451e+30	731	9440	9909	(39.70s)
ml-1/L1TiP9P55-8	1.101e+521	18098	23210	5834	(51.87s)
psy-1/j2-9yymHfeU	7.463e+1004	42186	50773	8805	(192.97s)
psy-2/uNrB2uG_9gs	7.749e+32	585	7994	8442	(24.06s)
speaking/KL_I9eE2aGk	9.156e+33	829	7766	8356	(23.13s)
theory_of_computation/aVv9WXwW95w	3.123e+41	578	5946	6347	(13.10s)
urban_energy_systems_and_policy/3WXWESx6jWY	3.759e+31	681	7931	8473

replace weight = 1
```text
22.11s  (edit calculation)
5.87s  (total: 27.98s)
anat-1/LQ7IXyQSXBk 8401 9013 1125 79,583,600,366,426,881,802,835,787,195,799,399,279,801,310,191,372,206,080,000
7.66s  (edit calculation)
2.23s  (total: 9.89s)
anat-2/IvK-UGOI5ZQ 4670 5869 1258 18,479,380,687,005,339,424,243,236,069,859,012,372,168,704,000,000,000,000
26.92s  (edit calculation)
7.24s  (total: 34.16s)
bio-1/pUH2_SGnm9g 9702 9961 359 123,527,442,364,876,800
22.69s  (edit calculation)
6.15s  (total: 28.84s)
bio-2/dEh_Zk9d0sE 8989 9165 346 4,150,807,400,323,584,000,000
11.60s  (edit calculation)
3.28s  (total: 14.88s)
bio-3/u7AnKjGOEgI 6281 6622 950 1,655,680,716,354,823,085,445,489,759,892,303,072,537,960,813,834,744,627,200,000,000,000
10.86s  (edit calculation)
2.97s  (total: 13.84s)
bio-4/gb0arNyiLCI 6061 6368 708 333,614,006,589,029,019,549,696,000,000,000
17.55s  (edit calculation)
4.93s  (total: 22.47s)
dental/uLz-Ur_Drk4 7509 8332 875 654,789,604,000,387,950,522,671,384,494,080,000,000
12.18s  (edit calculation)
3.79s  (total: 15.97s)
intmd_microeconomics/rhOpABGewYA 5681 7616 2085 17,548,316,564,154,032,899,187,037,065,818,614,735,793,443,410,057,941,076,172,654,106,409,946,158,953,646,299,945,736,601,600,000,000,000
17.11s  (edit calculation)
4.63s  (total: 21.74s)
intro_to_cs_programming_python/tsMrUdynbQQ 7655 8072 441 301,739,247,506,721,024,000
16.00s  (edit calculation)
4.35s  (total: 20.35s)
intro_to_neural_computation/K1pxJVdqlxw 7290 7914 873 812,200,914,332,677,518,897,721,481,563,768,012,800,000,000
25.84s  (edit calculation)
6.94s  (total: 32.78s)
intro_to_physics_20century/Fce9xf_kjyc 9440 9909 731 1,450,555,294,627,444,961,420,974,080,000
38.08s  (edit calculation)
12.91s  (total: 50.99s)
ml-1/L1TiP9P55-8 23210 5834 18098 110,085,621,681,126,156,385,521,320,120,965,120,567,767,903,388,704,131,079,434,865,801,008,680,715,281,824,853,294,725,142,869,457,691,943,320,911,992,342,107,721,681,936,760,790,701,560,760,689,452,915,497,143,625,213,255,201,617,484,130,097,666,837,446,519,453,083,699,687,503,137,699,985,096,966,783,052,542,680,681,996,911,711,508,120,863,301,097,214,986,890,413,382,081,191,593,102,741,261,811,113,702,041,245,179,406,857,986,474,297,710,710,381,783,381,375,464,360,787,070,653,705,438,475,304,728,692,721,901,776,645,493,605,848,844,839,917,109,592,497,744,422,205,869,565,142,210,474,550,460,887,959,745,337,507,895,447,093,373,980,114,944,000,000,000,000,000,000,000,000,000,000,000,000
126.11s  (edit calculation)
41.52s  (total: 167.62s)
psy-1/j2-9yymHfeU 50773 8805 42186 746,253,377,974,023,806,968,150,410,524,123,794,672,773,280,480,011,310,256,716,167,172,937,758,933,695,708,416,622,163,253,084,364,860,063,054,522,562,460,693,048,590,305,256,659,524,363,737,944,180,461,872,127,173,739,268,185,660,297,144,112,315,576,235,042,319,507,601,138,367,135,440,354,032,462,134,969,175,595,611,273,238,724,254,935,015,324,710,847,769,264,593,189,306,459,752,681,119,864,890,820,183,133,458,987,768,968,529,711,302,953,170,236,523,623,806,956,456,090,991,059,116,262,142,613,825,779,911,445,323,951,580,690,336,284,488,981,932,600,888,988,294,952,785,296,120,685,709,046,728,258,799,498,576,836,470,980,460,309,307,018,014,546,309,199,250,015,655,027,430,578,501,953,586,462,277,954,427,054,806,489,940,981,275,191,610,859,525,016,657,139,756,901,324,952,877,793,490,801,062,219,589,089,283,433,941,025,841,761,293,895,184,189,453,352,854,700,307,276,483,410,793,643,304,543,044,236,932,715,728,938,644,328,416,285,644,648,186,473,488,556,172,524,170,796,684,138,085,665,936,610,195,735,358,367,789,791,613,242,705,341,046,272,410,772,288,881,630,325,745,479,463,628,273,229,359,233,269,883,742,555,457,099,062,514,113,476,970,937,589,396,837,538,547,873,413,915,981,618,981,404,705,752,317,602,609,811,633,147,141,408,912,689,515,507,376,726,690,024,538,714,931,200,000,000,000,000,000,000,000,000,000,000,000,000
20.10s  (edit calculation)
5.15s  (total: 25.25s)
psy-2/uNrB2uG_9gs 7994 8442 585 774,850,350,759,510,537,313,072,128,000,000
18.11s  (edit calculation)
5.28s  (total: 23.39s)
speaking/KL_I9eE2aGk 7766 8356 829 9,156,045,332,497,099,087,449,629,392,896,000
10.37s  (edit calculation)
2.86s  (total: 13.22s)
theory_of_computation/aVv9WXwW95w 5946 6347 578 312,317,863,555,754,933,755,880,117,852,335,767,552,000
18.47s  (edit calculation)
5.07s  (total: 23.55s)
urban_energy_systems_and_policy/3WXWESx6jWY 7931 8473 681 37,587,254,825,598,424,927,174,656,000,000
```

replace weight = 2
```
21.74s  (edit calculation)
6.05s  (total: 27.79s)
anat-1/LQ7IXyQSXBk 8401 9013 1224 608,261,884,108,331,939,205,507,750,477,100,578,052,157,111,725,554,781,448,178,865,674,703,209,235,670,158,540,800,000,000,000,000,000
7.66s  (edit calculation)
2.22s  (total: 9.87s)
anat-2/IvK-UGOI5ZQ 4670 5869 1285 3,344,837,658,079,890,809,703,756,416,080,331,068,937,805,304,524,941,033,472,000,000,000
26.78s  (edit calculation)
7.32s  (total: 34.10s)
bio-1/pUH2_SGnm9g 9702 9961 377 1,029,381,864,411,876,019,200,000
22.90s  (edit calculation)
6.17s  (total: 29.06s)
bio-2/dEh_Zk9d0sE 8989 9165 368 13,998,175,605,464,064,291,328,896,000,000
11.81s  (edit calculation)
3.52s  (total: 15.34s)
bio-3/u7AnKjGOEgI 6281 6622 1037 334,310,493,913,411,067,363,764,955,583,161,054,299,275,797,709,250,157,586,792,873,773,533,180,723,200,000,000
10.86s  (edit calculation)
3.49s  (total: 14.35s)
bio-4/gb0arNyiLCI 6061 6368 735 11,570,900,337,631,430,178,918,711,522,416,598,528,000,000,000
22.80s  (edit calculation)
5.60s  (total: 28.40s)
dental/uLz-Ur_Drk4 7509 8332 897 169,064,120,753,825,447,252,576,540,408,284,009,267,200,000,000
13.84s  (edit calculation)
4.30s  (total: 18.14s)
intmd_microeconomics/rhOpABGewYA 5681 7616 2175 314,467,423,949,345,864,289,970,397,912,766,002,702,360,939,443,272,474,986,147,911,025,121,841,417,890,718,047,373,350,630,314,064,415,302,799,216,921,804,800,000,000,000,000,000,000
20.04s  (edit calculation)
5.34s  (total: 25.39s)
intro_to_cs_programming_python/tsMrUdynbQQ 7655 8072 453 26,947,072,823,639,429,260,800,000
20.00s  (edit calculation)
5.16s  (total: 25.16s)
intro_to_neural_computation/K1pxJVdqlxw 7290 7914 904 820,248,703,096,862,060,713,256,008,413,818,823,826,187,118,182,400,000,000
26.34s  (edit calculation)
7.03s  (total: 33.37s)
intro_to_physics_20century/Fce9xf_kjyc 9440 9909 769 180,141,627,950,256,334,308,049,615,610,051,913,960,000,000,000
42.36s  (edit calculation)
13.21s  (total: 55.57s)
ml-1/L1TiP9P55-8 23210 5834 18396 6,564,544,740,650,038,705,240,985,702,052,822,627,826,018,280,860,852,026,786,914,353,686,855,129,280,877,188,669,231,326,473,443,910,067,498,510,034,077,624,306,184,613,246,851,762,146,946,781,263,324,032,563,627,980,863,251,775,283,151,249,396,404,022,266,868,438,490,185,686,118,818,036,509,473,819,639,985,454,077,700,648,721,736,905,709,561,893,096,574,183,863,344,271,773,799,229,037,579,850,403,058,633,157,834,526,677,865,473,120,687,282,070,196,406,705,211,456,531,074,105,959,933,357,105,783,580,361,223,704,360,462,593,842,962,974,342,939,021,631,962,888,388,324,912,968,921,675,374,536,329,046,510,193,042,679,565,982,226,505,600,843,002,527,627,472,546,113,258,286,425,066,425,026,258,807,811,576,067,218,166,546,295,685,120,000,000,000,000,000,000,000,000,000,000,000,000,000,000,000,000,000,000,000
```

In [ ]:
sols = {}
for sbj in transcripts_asr.keys():
    t_asr = {k: v["asr"] for k, v in transcripts_asr[sbj].items()}
    t_anntd = {k: v["en"] for k, v in transcripts_anntd[sbj].items()}
    cur_sol = Solution()
    _t1 = time.perf_counter()
    min_dist = cur_sol.min_distance(' '.join(t_asr.values()), ' '.join(t_anntd.values()))
    _t2 = time.perf_counter()
    print(f"{_t2-_t1:.2f}s  (edit calculation)")
    path_cnt = cur_sol.count_paths()
    _t3 = time.perf_counter()
    print(f"{_t3-_t2:.2f}s  (total: {_t3-_t1:.2f}s)")
    sols[sbj] = cur_sol
    print(sbj, cur_sol.M, cur_sol.N, min_dist, f"{path_cnt:,}")

21.74s  (edit calculation)
6.05s  (total: 27.79s)
anat-1/LQ7IXyQSXBk 8401 9013 1224 608,261,884,108,331,939,205,507,750,477,100,578,052,157,111,725,554,781,448,178,865,674,703,209,235,670,158,540,800,000,000,000,000,000
7.66s  (edit calculation)
2.22s  (total: 9.87s)
anat-2/IvK-UGOI5ZQ 4670 5869 1285 3,344,837,658,079,890,809,703,756,416,080,331,068,937,805,304,524,941,033,472,000,000,000
26.78s  (edit calculation)
7.32s  (total: 34.10s)
bio-1/pUH2_SGnm9g 9702 9961 377 1,029,381,864,411,876,019,200,000
22.90s  (edit calculation)
6.17s  (total: 29.06s)
bio-2/dEh_Zk9d0sE 8989 9165 368 13,998,175,605,464,064,291,328,896,000,000
11.81s  (edit calculation)
3.52s  (total: 15.34s)
bio-3/u7AnKjGOEgI 6281 6622 1037 334,310,493,913,411,067,363,764,955,583,161,054,299,275,797,709,250,157,586,792,873,773,533,180,723,200,000,000
10.86s  (edit calculation)
3.49s  (total: 14.35s)
bio-4/gb0arNyiLCI 6061 6368 735 11,570,900,337,631,430,178,918,711,522,416,598,528,000,000,000
22.80s  (edit calculation)


---
---

fin